In [ ]:
import pandas as pd

df = pd.read_csv("./NOTEEVENTS.csv.gz", compression="gzip")
df.head()

In [ ]:
import pandas as pd

df = pd.read_csv(
    "./NOTEEVENTS.csv.gz",
    usecols=["TEXT"],
    nrows=5,
)

for i, note in enumerate(df["TEXT"], start=1):
    print(f"\n--- 第 {i} 条病历 ---\n")
    print(note[:1000])  

In [ ]:
FILE_PATH = r"./NOTEEVENTS.csv.gz"
OUT_INTERIM = r"./mimic_notes_interim.csv"      
SAMPLE_NROWS = 20000  
RANDOM_SEED = 42

SENTENCET5_MODEL = "sentence-transformers/sentence-t5-base"
SEM_SIM_THRESHOLD = 0.90   
BATCH_SIZE_EMBED = 64      
INFO_DENSITY_PERCENTILE = 10  

In [ ]:
import pandas as pd

usecols = ["CATEGORY", "TEXT"]
if SAMPLE_NROWS:
    df = pd.read_csv(FILE_PATH, usecols=usecols, nrows=SAMPLE_NROWS)
else:
    df = pd.read_csv(FILE_PATH, usecols=usecols)

print("原始记录总数:", len(df))
notes = df[df["CATEGORY"] == "Discharge summary"]["TEXT"].dropna().reset_index(drop=True)
print("筛选后（Discharge summary）数量:", len(notes))

notes.to_csv(OUT_INTERIM, index=False, header=True)

In [ ]:
import re
import pandas as pd

def deidentify_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\[\*\*\d{4}-\d{1,2}-\d{1,2}\*\*\]", " <DATE> ", text)
    text = re.sub(r"\[\*\*.*?\*\*\]", " <INFO> ", text)
    text = re.sub(r"\r", " ", text)
    text = re.sub(r"\t", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()

if isinstance(OUT_INTERIM, str):
    notes = pd.read_csv(OUT_INTERIM)
    if notes.shape[1] == 1:
        notes = notes.squeeze("columns")

cleaned = notes.apply(deidentify_text)

print("示例清洗后（前1条）:\n", cleaned.iloc[0][:500])

cleaned.to_csv(OUT_INTERIM.replace(".csv", "_deid.csv"), index=False, header=True)

In [ ]:
import re
def remove_template_sections(text: str) -> str:
    text = re.split(r"(Dictated By:|Signed By:|Dictation:|Job#:|Physician Signature:)", text, maxsplit=1)[0]
    new_text = re.sub(r"-{2,}", "", text)
    new_text = re.sub(r"_{2,}", "", new_text)
    new_text = re.sub(r"\n{2,}", "\n", new_text)
    return new_text.strip()

cleaned2 = cleaned.apply(remove_template_sections)
print("去模板后示例（前1条）:\n", cleaned2.iloc[0][:500])

cleaned2.to_csv(OUT_INTERIM.replace(".csv", "_deid_dehead.csv"), index=False, header=True)

In [ ]:
import pandas as pd
import re

notes = pd.read_csv("./mimic_notes_interim_deid_dehead.csv")

if notes.shape[1] == 1:
    notes = notes.squeeze("columns")
    
pattern_sign = notes.str.contains(r"(Dictated By:|Signed By:|Job#:|Physician Signature:)", regex=True, case=False).sum()
has_underscore = notes.str.contains(r"-{2,}|_{2,}", regex=True).sum()
has_multiple_newlines = notes.str.contains(r"\n{2,}", regex=True).sum()

lengths = notes.str.len()
print("平均长度:", round(lengths.mean(), 2))
print("最短:", lengths.min(), "最长:", lengths.max())
print("仍含模板签名:", pattern_sign)
print("包含下划线的病历数:", has_underscore)       
print("包含连续空行的病历数:", has_multiple_newlines) 

for i in range(3):
    print("\n--- 病历样例 ---\n")
    print(notes.sample(1).values[0][:800])

In [ ]:
import pandas as pd
import numpy as np

df_step3 = pd.read_csv(OUT_INTERIM.replace(".csv", "_deid_dehead.csv"))

if df_step3.shape[1] == 1:
    df_step3 = df_step3.squeeze("columns")

before = len(df_step3)
df_nodup = df_step3.drop_duplicates().reset_index(drop=True)
after = len(df_nodup)
print(f"精确去重：{before} -> {after}（去掉 {before - after} 条）")

lengths = df_nodup.apply(lambda x: len(str(x).split()))
print("平均单词数：", np.mean(lengths), "中位数：", np.median(lengths))

df_nodup.to_csv(OUT_INTERIM.replace(".csv", "_deid_dehead_nodup.csv"), index=False, header=True)

In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch
import math
import tqdm
import pandas as pd

OUT_INTERIM = "./my_.csv"  
SENTENCET5_MODEL = "all-MiniLM-L6-v2"  
BATCH_SIZE_EMBED = 64  
SEM_SIM_THRESHOLD = 0.9  

df = pd.read_csv("./mimic_notes_interim_deid_dehead_nodup.csv")

if df.shape[1] == 1:
    texts = df.squeeze("columns").dropna().astype(str).tolist()
else:
    texts = df.iloc[:, 0].dropna().astype(str).tolist()

print("待做语义去重文本数：", len(texts))

print(f"🧠 正在加载模型: {SENTENCET5_MODEL}")
model = SentenceTransformer(SENTENCET5_MODEL)
print("✅ 模型加载成功！")

embeddings = []
for i in tqdm.tqdm(range(0, len(texts), BATCH_SIZE_EMBED), desc="Encoding batches"):
    batch = texts[i:i + BATCH_SIZE_EMBED]
    emb = model.encode(batch, convert_to_tensor=True, show_progress_bar=False, device='cuda' if torch.cuda.is_available() else 'cpu')
    embeddings.append(emb)
embeddings = torch.cat(embeddings, dim=0)

keep_idx = []
seen = torch.zeros(len(texts), dtype=torch.bool)

for i in tqdm.tqdm(range(len(texts)), desc="Semantic dedup (greedy)"):
    if seen[i]:
        continue
    keep_idx.append(i)
    sims = util.cos_sim(embeddings[i], embeddings).squeeze(0)
    sim_mask = (sims > SEM_SIM_THRESHOLD).cpu().numpy()
    seen[sim_mask] = True 

kept_texts = [texts[i] for i in keep_idx]
print(f"语义去重后剩余：{len(kept_texts)} / {len(texts)}")

output_path = OUT_INTERIM.replace(".csv", "_deid_dehead_nodup_semfilter1.csv")
pd.Series(kept_texts).to_csv(output_path, index=False, header=True)
print("✅ 已保存文件到：", output_path)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("./my__deid_dehead_nodup_semfilter1.csv")

if df.shape[1] == 1:
    texts = df.iloc[:, 0].dropna().astype(str)
else:
    texts = df.dropna().astype(str)

print("语义去重后样本数：", len(texts))

lengths = texts.str.len()
print(f"平均长度: {np.mean(lengths):.1f}")
print(f"最短: {np.min(lengths)}, 最长: {np.max(lengths)}")

dup_count = texts.duplicated().sum()
print(f"仍有完全重复文本: {dup_count}")

for i in range(3):
    print("\n--- 样本 ---\n", texts.sample(1).values[0][:500])

In [ ]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import models, SentenceTransformer
from tqdm import tqdm

INPUT_FILE = "./my__deid_dehead_nodup_semfilter1.csv"
OUTPUT_FILE = "./mimic_notes_final_clean.csv"
MODEL_NAME = "all-MiniLM-L6-v2"  
BATCH_SIZE = 64
FILTER_PERCENTILE = 10 

df = pd.read_csv(INPUT_FILE)

if df.shape[1] == 1:
    texts = df.iloc[:, 0].dropna().astype(str).tolist()
else:
    texts = df.select_dtypes(include=["object"]).stack().dropna().astype(str).tolist()

print(f"🧾 已载入 {len(texts)} 条病历文本")

print(f"🧠 正在加载模型：{MODEL_NAME}")
word_emb = models.Transformer(MODEL_NAME)
pooling = models.Pooling(
    word_emb.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True,
    pooling_mode_cls_token=False,
    pooling_mode_max_tokens=False
)
model = SentenceTransformer(modules=[word_emb, pooling])
print("✅ 模型加载完成！")

embeddings = []
for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Encoding batches"):
    batch = texts[i:i + BATCH_SIZE]
    emb = model.encode(batch, convert_to_tensor=True, show_progress_bar=False, normalize_embeddings=False)
    embeddings.append(emb)

embeddings = torch.cat(embeddings, dim=0)
print(f"✅ 编码完成，共获得 {embeddings.shape[0]} 个向量")

norms = torch.norm(embeddings, dim=1).cpu().numpy()
print(f"📊 平均信息密度: {np.mean(norms):.4f}")
print(f"最小: {np.min(norms):.4f}, 最大: {np.max(norms):.4f}")

threshold = np.percentile(norms, FILTER_PERCENTILE)
mask = norms > threshold
filtered_texts = [t for t, keep in zip(texts, mask) if keep]

print(f"🧹 信息密度筛选后剩余: {len(filtered_texts)} / {len(texts)} "
      f"({len(filtered_texts)/len(texts)*100:.2f}%)")

filtered_texts = [t for t in filtered_texts if len(t.split()) > 50]

pd.Series(filtered_texts).to_csv(OUTPUT_FILE, index=False, header=True)
print(f"✅ 最终筛选结果已保存至：{OUTPUT_FILE}")

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("./mimic_notes_final_clean.csv")
texts = df.iloc[:, 0].dropna().astype(str)

print(f"🧾 最终文本数量：{len(texts)}")
print(f"平均长度：{np.mean(texts.str.len()):.1f}")
print(f"最短：{np.min(texts.str.len())}, 最长：{np.max(texts.str.len())}")

for i in range(3):
    sample = texts.sample(1).values[0]
    print(f"\n📄 示例 {i+1}:\n{sample[:500]}")